In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np


In [2]:
# SETUP

TRAINING_DATA = "~/Downloads/training_data_clean.csv"


# def prep_data():
df = pd.read_csv(TRAINING_DATA)

# rename
df.columns = [
    'id',
    'best_tasks_free',
    'acad_tasks_rating',
    'best_tasks_select',
    'subopt_freq_rating',
    'subopt_tasks_select',
    'subopt_tasks_free',
    'evidence_freq_rating',
    'verify_freq_rating',
    'verify_method_free',
    'target'
    ]

for task in df['verify_freq_rating'].unique():
    print(task)

# prep_data()


4 — Often
5 — Very often
3 — Sometimes
2 — Rarely
1 — Never
nan


In [3]:
def split_data(df):
    students = df['id'].unique()
    train_idx, test_idx = train_test_split(
        students, test_size=0.3, random_state=22
    )

    train_df = df[df['id'].isin(train_idx)]
    test_df = df[df['id'].isin(test_idx)]

    train_groups = train_df['id'].values
    test_groups = test_df['id'].values

    x_train = train_df.drop(columns=['target', 'id'])
    y_train = train_df['target']

    x_test = test_df.drop(columns=['target', 'id'])
    y_test = test_df['target']

    return x_train, x_test, y_train, y_test, train_groups, test_groups

In [4]:
# EXPLORE DATA

df.isnull().sum()

id                       0
best_tasks_free          2
acad_tasks_rating        0
best_tasks_select       15
subopt_freq_rating       1
subopt_tasks_select     22
subopt_tasks_free       11
evidence_freq_rating    62
verify_freq_rating       4
verify_method_free      10
target                   0
dtype: int64

In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer, OrdinalEncoder
from typing import Dict, List
import unicodedata



text_cols = [
    'best_tasks_free',
    'subopt_tasks_free',
    'verify_method_free',
    ]


ord_cols = [
    'acad_tasks_rating',
    'subopt_freq_rating',
    'evidence_freq_rating',
    'verify_freq_rating',
    ]

likert_categories = [
    '1 — Not at all likely',
    '2 — Unlikely',
    '3 — Neutral / Unsure',
    '4 — Likely',
    '5 — Very likely'
]

freq_categories = [
    '1 — Never',
    '2 — Rarely',
    '3 — Sometimes',
    '4 — Often',
    '5 — Very often'
]

ord_categories = [
    likert_categories,  # acad_tasks_rating
    freq_categories,    # subopt_freq_rating
    freq_categories,    # evidence_freq_rating
    freq_categories,    # verify_freq_rating
]

cat_cols = [
    'best_tasks_select',
    'subopt_tasks_select',
    ]

cat_multi_select_choices = [
    "Explaining complex concepts simply",
    "Drafting professional text (e.g., emails, résumés)",
    "Brainstorming or generating creative ideas",
    "Writing or editing essays/reports",
    "Converting content between formats (e.g., LaTeX)",
    "Writing or debugging code",
    "Data processing or analysis",
    "Math computations",
]

# custom function makecorpus just to keep consistency in pipeline
class MakeCorpus(BaseEstimator, TransformerMixin):
    # required by TansformerMixin -ignore
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        # X is DataFrame with text columns
        return X.agg(' '.join, axis=1).tolist()
    


def preprocess_text():
    # TFIDF manually (classical)
    return Pipeline([
        ('imputer', SimpleImputer(strategy="constant", fill_value="")),
        ('combiner', MakeCorpus()),
        # hardcoded stop_words param for now
        # TODO: try with max instead
        ('tfidf', TfidfVectorizer(stop_words='english'))
    ])

ord_encoder = OrdinalEncoder(
    categories=ord_categories,
    handle_unknown='use_encoded_value',
    unknown_value=np.nan
)


def preprocess_ord():
    

    return Pipeline([
        ('enc', ord_encoder),
        ('imp', KNNImputer(n_neighbors=3)),
    ])

def _normalize(s: str) -> str:
    # normalize unicode, collapse spaces, strip
    s = unicodedata.normalize("NFKC", str(s))
    s = " ".join(s.split())
    return s.strip()

def _split_top_level_commas(s: str) -> List[str]:
    """
    Split on commas that are NOT inside parentheses.
    Example: "Drafting ... (e.g., emails, résumés), Math computations"
    -> ["Drafting ... (e.g., emails, résumés)", "Math computations"]
    """
    parts = []
    buf = []
    depth = 0
    for ch in s:
        if ch == '(':
            depth += 1
            buf.append(ch)
        elif ch == ')':
            depth = max(0, depth - 1)
            buf.append(ch)
        elif ch == ',' and depth == 0:
            parts.append("".join(buf))
            buf = []
        else:
            buf.append(ch)
    if buf:
        parts.append("".join(buf))
    return parts

class MultiSelectBinarizerPerColumn(BaseEstimator, TransformerMixin):
    """
    One-hot encode multi-select columns using a safe split and a fixed
    set of canonical choices. Clone-safe: __init__ does not modify params.
    If the original cell is NaN -> all dummies for that column are NaN.
    """
    def __init__(self, classes: List[str]):
        self.classes = classes  # DO NOT MODIFY HERE (clone-safe)

    # internal helpers use fitted attributes
    def _parse_cell(self, x) -> List[str]:
        if pd.isna(x) or (isinstance(x, str) and x.strip() == ""):
            return []
        norm = _normalize(x)
        pieces = [_normalize(p) for p in _split_top_level_commas(norm)]
        # keep only exact matches (normalized)
        return [p for p in pieces if p in self.classes_]

    def fit(self, X, y=None):
        X = pd.DataFrame(X)

        # create normalized, fitted classes (separate from init param)
        self.classes_ = [_normalize(c) for c in self.classes]

        self.mlbs_: Dict[str, MultiLabelBinarizer] = {}
        self.col_to_outcols_: Dict[str, List[str]] = {}

        for col in X.columns:
            mlb = MultiLabelBinarizer(classes=self.classes_)
            mlb.fit([[]])  # fit on fixed classes only
            self.mlbs_[col] = mlb
            self.col_to_outcols_[col] = [f"{col}__{c}" for c in mlb.classes_]
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        blocks = []
        for col in X.columns:
            mlb = self.mlbs_[col]
            outcols = self.col_to_outcols_[col]
            is_missing = X[col].isna()
            lists = X[col].apply(self._parse_cell)
            arr = mlb.transform(lists)
            df_bin = pd.DataFrame(arr, columns=outcols, index=X.index)
            df_bin.loc[is_missing, :] = np.nan  # preserve missingness for KNN later
            blocks.append(df_bin)
        return pd.concat(blocks, axis=1)

class MultiSelectSummary(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = pd.DataFrame(X)
        df = pd.DataFrame()
        df['num_selected'] = X.apply(lambda row: sum([
            isinstance(v, str) and v.strip() != "" for v in row
        ]), axis=1)
        return df

def preprocess_cat():
    return make_pipeline(MultiSelectBinarizerPerColumn(classes=cat_multi_select_choices),
                                 SimpleImputer(strategy="constant", fill_value=0))

def create_preprocessor():
    # create pipelines for each type, applied per column within each subset
    # pipelines run in tandem
    # changed names to shorter versions for param_grid referencing
    transformers = [
        ("text", preprocess_text(), text_cols), # pass in just the names of the columns for now, df specified later in full pipeline
        ("ord", preprocess_ord(), ord_cols),
         ("cat", preprocess_cat(), cat_cols),
    ]

    return ColumnTransformer(transformers=transformers)

# Full pipeline
# added logistic regression as placeholder classifier for now
preprocessor = create_preprocessor()
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

# ============ GRID SEARCH ON FULL PIPELINE ============

param_grid = {
    # Text TF-IDF parameters

    'preprocessor__text__tfidf__max_features': [500, 1000, 2000],
    'preprocessor__text__tfidf__ngram_range': [(1, 1), (1, 2)],
    'preprocessor__text__tfidf__min_df': [2, 3, 4],
    'preprocessor__text__tfidf__max_df': [0.75, 0.9]


    # Classifier parameters
    # TODO: change to SVM or RandomForest and tune their hyperparams
}

# Grid search
grid_search = GridSearchCV(
    full_pipeline,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

# Fit on raw data (pipeline handles preprocessing)

x_train, x_test, y_train, y_test, train_group, test_groups = split_data(df)

full_pipeline.fit(x_train, y_train)

# y_pred = full_pipeline.predict(x_train)
# print(f"Train accuracy: {np.mean(y_pred == y_train)}")
# df specified here
grid_search.fit(x_train, y_train)

y_pred_train = grid_search.predict(x_train)
train_acc = np.mean(y_pred_train == y_train)
print(f"Grid Search - Training accuracy: {train_acc:.3f}")
print(f"Grid Search - Best CV score: {grid_search.best_score_:.3f}")
print(f"Best parameters:{grid_search.best_params_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
[CV] END preprocessor__text__tfidf__max_df=0.75, preprocessor__text__tfidf__max_features=500, preprocessor__text__tfidf__min_df=2, preprocessor__text__tfidf__ngram_range=(1, 1); total time=   0.2s
[CV] END preprocessor__text__tfidf__max_df=0.75, preprocessor__text__tfidf__max_features=500, preprocessor__text__tfidf__min_df=2, preprocessor__text__tfidf__ngram_range=(1, 1); total time=   0.2s
[CV] END preprocessor__text__tfidf__max_df=0.75, preprocessor__text__tfidf__max_features=500, preprocessor__text__tfidf__min_df=2, preprocessor__text__tfidf__ngram_range=(1, 1); total time=   0.2s
[CV] END preprocessor__text__tfidf__max_df=0.75, preprocessor__text__tfidf__max_features=500, preprocessor__text__tfidf__min_df=2, preprocessor__text__tfidf__ngram_range=(1, 1); total time=   0.2s
[CV] END preprocessor__text__tfidf__max_df=0.75, preprocessor__text__tfidf__max_features=500, preprocessor__text__tfidf__min_df=2, preprocessor__text_

Logistic Regression below

In [6]:
from sklearn.model_selection import GroupKFold


best_tfidf = grid_search.best_estimator_

grid_log_reg = {'classifier__C': [0.001, 0.01, 0.1, 1.]}
cv1 = GroupKFold(n_splits=5)
grid_search_logreg = GridSearchCV(best_tfidf, grid_log_reg,cv=cv1,scoring='accuracy', n_jobs=-1,verbose=2)
grid_search_logreg.fit(x_train, y_train, groups = train_group)

print(f"Best log reg model parameters: {grid_search_logreg.best_params_}")
print(f"Best log reg model: {grid_search_logreg.best_estimator_}")
print(f"Best log reg model score: {grid_search_logreg.best_score_}")
best_model = grid_search_logreg.best_estimator_
train_acc = best_model.score(x_train, y_train)
print(train_acc)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
[CV] END ................................classifier__C=0.001; total time=   0.1s
[CV] END ................................classifier__C=0.001; total time=   0.1s
[CV] END ................................classifier__C=0.001; total time=   0.1s
[CV] END ................................classifier__C=0.001; total time=   0.1s
[CV] END ................................classifier__C=0.001; total time=   0.1s
[CV] END .................................classifier__C=0.01; total time=   0.2s
[CV] END .................................classifier__C=0.01; total time=   0.2s
[CV] END .................................classifier__C=0.01; total time=   0.2s
[CV] END .................................classifier__C=0.01; total time=   0.2s
[CV] END .................................classifier__C=0.01; total time=   0.2s
[CV] END ..................................classifier__C=0.1; total time=   0.2s
[CV] END ..................................classi

In [7]:
# Trying linear SVC
from sklearn.svm import LinearSVC

best_preprocessor = best_tfidf.named_steps['preprocessor']

full_pipeline2 = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', LinearSVC(dual=True))
])

grid_lin_svc= {'classifier__C': [0.01, 0.1, 1.0, 10.0], 'classifier__max_iter': [5000, 10000, 20000]}
grid_search_lin_svc = GridSearchCV(full_pipeline2, grid_lin_svc,cv=cv1,scoring='accuracy', n_jobs=-1,verbose=2)
grid_search_lin_svc.fit(x_train, y_train, groups =train_group)

print(f"Best log reg model parameters: {grid_search_lin_svc.best_params_}")
print(f"Best log reg model: {grid_search_lin_svc.best_estimator_}")
print(f"Best log reg model score: {grid_search_lin_svc.best_score_}")
best_model_svc = grid_search_lin_svc.best_estimator_
train_acc_svc = best_model_svc.score(x_train, y_train)
print(train_acc_svc)




Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV] END ......classifier__C=0.01, classifier__max_iter=5000; total time=   0.1s
[CV] END ......classifier__C=0.01, classifier__max_iter=5000; total time=   0.1s
[CV] END ......classifier__C=0.01, classifier__max_iter=5000; total time=   0.1s
[CV] END ......classifier__C=0.01, classifier__max_iter=5000; total time=   0.1s
[CV] END ......classifier__C=0.01, classifier__max_iter=5000; total time=   0.1s
[CV] END .....classifier__C=0.01, classifier__max_iter=10000; total time=   0.1s
[CV] END .....classifier__C=0.01, classifier__max_iter=10000; total time=   0.1s
[CV] END .....classifier__C=0.01, classifier__max_iter=10000; total time=   0.1s
[CV] END .....classifier__C=0.01, classifier__max_iter=10000; total time=   0.2s
[CV] END .....classifier__C=0.01, classifier__max_iter=10000; total time=   0.2s
[CV] END .....classifier__C=0.01, classifier__max_iter=20000; total time=   0.2s
[CV] END .....classifier__C=0.01, classifier__ma

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV] END ......classifier__C=10.0, classifier__max_iter=5000; total time=   1.3s
[CV] END ......classifier__C=10.0, classifier__max_iter=5000; total time=   1.3s


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV] END ......classifier__C=10.0, classifier__max_iter=5000; total time=   1.2s
[CV] END ......classifier__C=10.0, classifier__max_iter=5000; total time=   1.3s
[CV] END .....classifier__C=10.0, classifier__max_iter=10000; total time=   1.6s
[CV] END .....classifier__C=10.0, classifier__max_iter=10000; total time=   1.8s
[CV] END .....classifier__C=10.0, classifier__max_iter=10000; total time=   1.7s
[CV] END .....classifier__C=10.0, classifier__max_iter=10000; total time=   1.8s
[CV] END .....classifier__C=10.0, classifier__max_iter=10000; total time=   1.1s
[CV] END ......classifier__C=10.0, classifier__max_iter=5000; total time=   1.1s


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV] END .....classifier__C=10.0, classifier__max_iter=20000; total time=   1.5s
[CV] END .....classifier__C=10.0, classifier__max_iter=20000; total time=   1.4s
[CV] END .....classifier__C=10.0, classifier__max_iter=20000; total time=   0.9s
[CV] END .....classifier__C=10.0, classifier__max_iter=20000; total time=   1.2s
[CV] END .....classifier__C=10.0, classifier__max_iter=20000; total time=   1.1s
Best log reg model parameters: {'classifier__C': 0.1, 'classifier__max_iter': 5000}
Best log reg model: Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('text',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='',
                                                                                 strategy='constant')),
                                                                  ('combiner',
                                           

In [8]:
# Trying Naive Bayes

from sklearn.naive_bayes import MultinomialNB

full_pipeline3 = Pipeline([
    ('preprocessor', best_preprocessor),
    ('classifier', MultinomialNB())
])

grid_lin_nb=  {'classifier__alpha': [0.1, 0.5, 1.0, 2.0]}
grid_search_nb = GridSearchCV(full_pipeline3, grid_lin_nb,cv=cv1,scoring='accuracy', n_jobs=-1,verbose=2)
grid_search_nb.fit(x_train, y_train, groups=train_group)

print(f"Best log reg model parameters: {grid_search_nb.best_params_}")
print(f"Best log reg model: {grid_search_nb.best_estimator_}")
print(f"Best log reg model score: {grid_search_nb.best_score_}")
best_model_nb = grid_search_nb.best_estimator_
train_acc_nb = grid_search_nb.score(x_train, y_train)
print(train_acc_nb)


Fitting 5 folds for each of 4 candidates, totalling 20 fits
[CV] END ..............................classifier__alpha=0.1; total time=   0.1s
[CV] END ..............................classifier__alpha=0.1; total time=   0.1s
[CV] END ..............................classifier__alpha=0.1; total time=   0.1s
[CV] END ..............................classifier__alpha=0.1; total time=   0.1s
[CV] END ..............................classifier__alpha=0.1; total time=   0.1s
[CV] END ..............................classifier__alpha=0.5; total time=   0.1s
[CV] END ..............................classifier__alpha=0.5; total time=   0.1s
[CV] END ..............................classifier__alpha=0.5; total time=   0.1s
[CV] END ..............................classifier__alpha=0.5; total time=   0.2s
[CV] END ..............................classifier__alpha=0.5; total time=   0.2s
[CV] END ..............................classifier__alpha=1.0; total time=   0.2s
[CV] END ..............................classifier

In [9]:
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import GridSearchCV
import numpy as np

# Assuming you already have these from your RF tuning:
# best_tfidf_rf = grid_search_rf.best_estimator_
# best_preprocessor = best_tfidf_rf.named_steps['preprocessor']

# Helper to convert sparse -> dense for the NN
to_dense = FunctionTransformer(
    lambda X: X.toarray() if hasattr(X, "toarray") else X,
    accept_sparse=True
)

nn_pipeline = Pipeline([
    ('preprocessor', best_preprocessor),   # your ColumnTransformer with tfidf + others
    ('to_dense', to_dense),               # convert sparse to dense
    ('classifier', MLPClassifier(
        max_iter=200,
        early_stopping=True,
        random_state=22
    ))
])
param_grid_nn = {
    # network size
    'classifier__hidden_layer_sizes': [
        (64,),
        (128,),
        (64, 32)
    ],
    # L2 regularization
    'classifier__alpha': [1e-4, 1e-3, 1e-2],
    # learning rate
    'classifier__learning_rate_init': [1e-3, 3e-3],
    # batch size
    'classifier__batch_size': [16, 32]
}
grid_search_nn = GridSearchCV(
    nn_pipeline,
    param_grid_nn,
    cv=cv1,                    # your GroupKFold or StratifiedGroupKFold
    scoring='accuracy',
    n_jobs=-1,                 # parallel over hyperparam configs
    verbose=2
)

grid_search_nn.fit(x_train, y_train, groups=train_group)

print("Best NN params:", grid_search_nn.best_params_)
print("Best NN CV score:", grid_search_nn.best_score_)

best_model_nn = grid_search_nn.best_estimator_
train_acc_nn = best_model_nn.score(x_train, y_train)
print("NN train accuracy:", train_acc_nn)


Fitting 5 folds for each of 36 candidates, totalling 180 fits
[CV] END classifier__alpha=0.0001, classifier__batch_size=16, classifier__hidden_layer_sizes=(64,), classifier__learning_rate_init=0.001; total time=   1.6s
[CV] END classifier__alpha=0.0001, classifier__batch_size=16, classifier__hidden_layer_sizes=(64,), classifier__learning_rate_init=0.001; total time=   1.7s
[CV] END classifier__alpha=0.0001, classifier__batch_size=16, classifier__hidden_layer_sizes=(64,), classifier__learning_rate_init=0.001; total time=   1.8s
[CV] END classifier__alpha=0.0001, classifier__batch_size=16, classifier__hidden_layer_sizes=(64,), classifier__learning_rate_init=0.001; total time=   1.8s
[CV] END classifier__alpha=0.0001, classifier__batch_size=16, classifier__hidden_layer_sizes=(64,), classifier__learning_rate_init=0.001; total time=   1.8s
[CV] END classifier__alpha=0.0001, classifier__batch_size=16, classifier__hidden_layer_sizes=(64,), classifier__learning_rate_init=0.003; total time=   1

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline, Pipeline, FeatureUnion
from sklearn.preprocessing import OrdinalEncoder, FunctionTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.preprocessing import MultiLabelBinarizer
from typing import Dict, List
import unicodedata
from sklearn.svm import LinearSVC


class ColumnExtractor(BaseEstimator, TransformerMixin):
   """Extract a single column and convert to list of strings"""
  
   def __init__(self, column_index):
       self.column_index = column_index
  
   def fit(self, X, y=None):
       return self
  
   def transform(self, X):
       # X is a DataFrame with text columns
       X = pd.DataFrame(X)
       col = X.iloc[:, self.column_index]
       # Convert to list of strings, handling NaN
       return [str(x) if pd.notna(x) else '' for x in col]




def preprocess_text_separate():
   """
   Create SEPARATE TF-IDF vectorizers for each text column
   This allows different vocabulary importance per question
   """
   return FeatureUnion([
       # best_tasks_free - most informative (what tasks?)
       ('best_tasks', Pipeline([
           ('extract', ColumnExtractor(column_index=0)),
           ('tfidf', TfidfVectorizer(
               max_features=800,
               ngram_range=(1, 2),
               min_df=2,
               max_df=0.75,
               stop_words='english'
           ))
       ])),
      
       # subopt_tasks_free - medium informative (suboptimal responses?)
       ('subopt_tasks', Pipeline([
           ('extract', ColumnExtractor(column_index=1)),
           ('tfidf', TfidfVectorizer(
               max_features=400,
               ngram_range=(1, 2),
               min_df=2,
               max_df=0.75,
               stop_words='english'
           ))
       ])),
      
       # verify_method_free - medium informative (how verify?)
       ('verify_method', Pipeline([
           ('extract', ColumnExtractor(column_index=2)),
           ('tfidf', TfidfVectorizer(
               max_features=400,
               ngram_range=(1, 2),
               min_df=2,
               max_df=0.75,
               stop_words='english'
           ))
       ])),
   ])


Generating best classfier for MLP/NN

In [19]:
def create_preprocessor2():
   transformers = [
       ("text", preprocess_text_separate(), text_cols), 
       ("ord", preprocess_ord(), ord_cols),
       ("cat", preprocess_cat(), cat_cols),
   ]


   return ColumnTransformer(transformers=transformers)


preprocessor = create_preprocessor2()

full_pipeline_nn = Pipeline([
   ("preprocessor", preprocessor),
   ('classifier', MLPClassifier(
        max_iter=200,
        early_stopping=True,
        random_state=22
    ))
])


param_grid = {
   # best_tasks_free
   "preprocessor__text__best_tasks__tfidf__max_features": [500, 800],
   "preprocessor__text__best_tasks__tfidf__ngram_range": [(1, 1), (1, 2)],


   # subopt_tasks_free
   "preprocessor__text__subopt_tasks__tfidf__max_features": [300, 500],
   "preprocessor__text__subopt_tasks__tfidf__ngram_range": [(1, 1), (1, 2)],


   # verify_method_free
   "preprocessor__text__verify_method__tfidf__max_features": [300, 500],
   "preprocessor__text__verify_method__tfidf__ngram_range": [(1, 1), (1, 2)],
}
grid_search_nn= GridSearchCV(
   estimator=full_pipeline_nn,
   param_grid=param_grid,
   cv=cv1,
   scoring="accuracy",
   n_jobs=-1,
   verbose=2
)
grid_search_nn.fit(x_train, y_train, groups=train_group)
print("Best params:", grid_search_nn.best_params_)
print("Best CV score:", grid_search_nn.best_score_)

Fitting 5 folds for each of 64 candidates, totalling 320 fits
[CV] END preprocessor__text__best_tasks__tfidf__max_features=500, preprocessor__text__best_tasks__tfidf__ngram_range=(1, 1), preprocessor__text__subopt_tasks__tfidf__max_features=300, preprocessor__text__subopt_tasks__tfidf__ngram_range=(1, 1), preprocessor__text__verify_method__tfidf__max_features=300, preprocessor__text__verify_method__tfidf__ngram_range=(1, 1); total time=   0.5s
[CV] END preprocessor__text__best_tasks__tfidf__max_features=500, preprocessor__text__best_tasks__tfidf__ngram_range=(1, 1), preprocessor__text__subopt_tasks__tfidf__max_features=300, preprocessor__text__subopt_tasks__tfidf__ngram_range=(1, 1), preprocessor__text__verify_method__tfidf__max_features=300, preprocessor__text__verify_method__tfidf__ngram_range=(1, 1); total time=   0.6s
[CV] END preprocessor__text__best_tasks__tfidf__max_features=500, preprocessor__text__best_tasks__tfidf__ngram_range=(1, 1), preprocessor__text__subopt_tasks__tfidf__

For random forest

In [20]:
from sklearn.ensemble import RandomForestClassifier



full_pipeline = Pipeline([
   ("preprocessor", preprocessor),
   ("clf", RandomForestClassifier(
       n_estimators=300,
       random_state=22,
       n_jobs=-1
   ))
])




grid_search = GridSearchCV(
   estimator=full_pipeline,
   param_grid=param_grid,
   cv=cv1,
   scoring="accuracy",
   n_jobs=-1,
   verbose=2
)


grid_search.fit(x_train, y_train, groups=train_group)


print("Best params:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)
best_model = grid_search.best_estimator_




Fitting 5 folds for each of 64 candidates, totalling 320 fits
[CV] END preprocessor__text__best_tasks__tfidf__max_features=500, preprocessor__text__best_tasks__tfidf__ngram_range=(1, 1), preprocessor__text__subopt_tasks__tfidf__max_features=300, preprocessor__text__subopt_tasks__tfidf__ngram_range=(1, 1), preprocessor__text__verify_method__tfidf__max_features=300, preprocessor__text__verify_method__tfidf__ngram_range=(1, 1); total time=   1.7s
[CV] END preprocessor__text__best_tasks__tfidf__max_features=500, preprocessor__text__best_tasks__tfidf__ngram_range=(1, 1), preprocessor__text__subopt_tasks__tfidf__max_features=300, preprocessor__text__subopt_tasks__tfidf__ngram_range=(1, 1), preprocessor__text__verify_method__tfidf__max_features=300, preprocessor__text__verify_method__tfidf__ngram_range=(1, 1); total time=   1.8s
[CV] END preprocessor__text__best_tasks__tfidf__max_features=500, preprocessor__text__best_tasks__tfidf__ngram_range=(1, 1), preprocessor__text__subopt_tasks__tfidf__

In [21]:
from sklearn.model_selection import RandomizedSearchCV

best_preprocessor =  grid_search.best_estimator_.named_steps['preprocessor']
# best_preprocessor =  best_tfidf.named_steps['preprocessor']
full_pipeline4 = Pipeline([
   ('preprocessor', best_preprocessor),
   ('classifier', RandomForestClassifier(random_state=22, bootstrap=True, n_jobs=-1))
])


grid_rf = {
   # Number of trees — broader, but still reasonable
   'classifier__n_estimators': [170, 180, 200],


   # Tree depth — wide range but avoids crazy huge trees
   'classifier__max_depth': [40, 50, 60],


   # Feature sampling strategy
   'classifier__max_features': ['sqrt',0.25, 0.5, 0.75, 1.0],


   # Minimum samples per split — helps control overfitting
   'classifier__min_samples_split': [2, 5, 10],


   # Minimum samples per leaf — stabilizes performance
   'classifier__min_samples_leaf': [1, 2, 4],


   # Bootstrap vs. not (big RF difference)
   'classifier__bootstrap': [True, False],
}


grid_rf2 = {
   # Number of trees — broader, but still reasonable
   'classifier__n_estimators': [171, 173, 174],


   # Tree depth — wide range but avoids crazy huge trees
   'classifier__max_depth': [ 42, 43, None],


   # Feature sampling strategy
   'classifier__max_features': ['sqrt'],


   # Minimum samples per split — helps control overfitting
   'classifier__min_samples_split': [2, 3, 4, 5],


   # Minimum samples per leaf — stabilizes performance
   'classifier__min_samples_leaf': [1, 2],


   # Bootstrap vs. not (big RF difference)
   'classifier__bootstrap': [True],
}



grid_search_rf = GridSearchCV(full_pipeline4, grid_rf2,cv=cv1,scoring='accuracy', n_jobs=-1,verbose=2)
# grid_search_rf = RandomizedSearchCV(
#     estimator=full_pipeline4,
#     param_distributions=grid_rf,
#     n_iter=200,        
#     cv=cv1,
#     n_jobs=2,
#     verbose=1
# )
grid_search_rf.fit(x_train, y_train, groups=train_group)


print(f"Best log reg model parameters: {grid_search_rf.best_params_}")
print(f"Best log reg model score: {grid_search_rf.best_score_}")
best_model_rf = grid_search_rf.best_estimator_
train_acc_rf = grid_search_rf.score(x_train, y_train)
print(train_acc_rf)

Fitting 5 folds for each of 72 candidates, totalling 360 fits
[CV] END classifier__bootstrap=True, classifier__max_depth=42, classifier__max_features=sqrt, classifier__min_samples_leaf=1, classifier__min_samples_split=2, classifier__n_estimators=171; total time=   0.5s
[CV] END classifier__bootstrap=True, classifier__max_depth=42, classifier__max_features=sqrt, classifier__min_samples_leaf=1, classifier__min_samples_split=2, classifier__n_estimators=171; total time=   0.5s
[CV] END classifier__bootstrap=True, classifier__max_depth=42, classifier__max_features=sqrt, classifier__min_samples_leaf=1, classifier__min_samples_split=2, classifier__n_estimators=171; total time=   0.5s
[CV] END classifier__bootstrap=True, classifier__max_depth=42, classifier__max_features=sqrt, classifier__min_samples_leaf=1, classifier__min_samples_split=2, classifier__n_estimators=171; total time=   0.5s
[CV] END classifier__bootstrap=True, classifier__max_depth=42, classifier__max_features=sqrt, classifier__

In [132]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Split train into train_small and validation
x_train_small, x_val, y_train_small, y_val = train_test_split(
    x_train, y_train, 
    test_size=0.15,      # 15% validation
    random_state=22,
    stratify=y_train     # important for classification
)

# Fit model normally
best_model_rf.fit(x_train_small, y_train_small)

# Predict validation performance
y_pred_val = best_model_rf.predict(x_val)

val_acc = accuracy_score(y_val, y_pred_val)
print("Validation accuracy:", val_acc)

Validation accuracy: 0.5747126436781609


This is tuning the tfidf vectorizer and rf sequentially

In [67]:
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier

preprocessor_rf = create_preprocessor()

base_pipeline = Pipeline([
    ('preprocessor', preprocessor_rf),
    ('classifier', RandomForestClassifier(
        random_state=22,
        n_jobs=-1,
        class_weight='balanced'
    )),
])



In [ ]:
def objective(trial):
    # ---- TF-IDF hyperparameters ----
    tfidf_max_features = trial.suggest_categorical(
        "tfidf_max_features", [500, 1000, 2000]
    )
    tfidf_ngram_range = trial.suggest_categorical(
        "tfidf_ngram_range", [(1, 1), (1, 2)]
    )
    tfidf_min_df = trial.suggest_int("tfidf_min_df", 2, 4)
    tfidf_max_df = trial.suggest_categorical("tfidf_max_df", [0.75, 0.9])

    # ---- RandomForest hyperparameters ----
    rf_n_estimators = trial.suggest_int("rf_n_estimators", 100, 500, step=100)
    rf_max_depth = trial.suggest_categorical("rf_max_depth", [None, 10, 20, 30])
    rf_max_features = trial.suggest_categorical("rf_max_features", ["sqrt", "log2"])
    rf_min_samples_split = trial.suggest_int("rf_min_samples_split", 2, 10, step=2)
    rf_min_samples_leaf = trial.suggest_int("rf_min_samples_leaf", 1, 4)

    pipe = clone(base_pipeline)

    # Set hyperparams
    pipe.set_params(
        preprocessor__text__tfidf__max_features=tfidf_max_features,
        preprocessor__text__tfidf__ngram_range=tfidf_ngram_range,
        preprocessor__text__tfidf__min_df=tfidf_min_df,
        preprocessor__text__tfidf__max_df=tfidf_max_df,
        classifier__n_estimators=rf_n_estimators,
        classifier__max_depth=rf_max_depth,
        classifier__max_features=rf_max_features,
        classifier__min_samples_split=rf_min_samples_split,
        classifier__min_samples_leaf=rf_min_samples_leaf,
    )

    # ☑️ Grouped CV
    scores = cross_val_score(
        pipe,
        x_train,
        y_train,
        cv=cv1,                 # your GroupKFold or StratifiedGroupKFold
        groups=train_group,     # <--- IMPORTANT
        scoring="accuracy",
        n_jobs=1                 # avoid MakeCorpus pickling issue
    )

    return scores.mean()


In [69]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=40)

print("Best CV score:", study.best_value)
print("Best params:", study.best_params)

best_params = study.best_params
best_pipeline = clone(base_pipeline)

best_pipeline.set_params(
    preprocessor__text__tfidf__max_features=best_params["tfidf_max_features"],
    preprocessor__text__tfidf__ngram_range=best_params["tfidf_ngram_range"],
    preprocessor__text__tfidf__min_df=best_params["tfidf_min_df"],
    preprocessor__text__tfidf__max_df=best_params["tfidf_max_df"],
    classifier__n_estimators=best_params["rf_n_estimators"],
    classifier__max_depth=best_params["rf_max_depth"],
    classifier__max_features=best_params["rf_max_features"],
    classifier__min_samples_split=best_params["rf_min_samples_split"],
    classifier__min_samples_leaf=best_params["rf_min_samples_leaf"],
)


[I 2025-11-19 18:27:48,077] A new study created in memory with name: no-name-fa75de96-4563-46bd-ab58-a1885273d173
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 1) which is of type tuple.
  warnings.warn(message)
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (1, 2) which is of type tuple.
  warnings.warn(message)
[I 2025-11-19 18:27:49,243] Trial 0 finished with value: 0.6443094916779126 and parameters: {'tfidf_max_features': 500, 'tfidf_ngram_range': (1, 2), 'tfidf_min_df': 4, 'tfidf_max_df': 0.9, 'rf_n_estimators': 100, 'rf_max_depth': 30, 'rf_max_features': 'sqrt', 'rf_min_samples_

Best CV score: 0.682366171839856
Best params: {'tfidf_max_features': 500, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 3, 'tfidf_max_df': 0.9, 'rf_n_estimators': 500, 'rf_max_depth': 10, 'rf_max_features': 'sqrt', 'rf_min_samples_split': 8, 'rf_min_samples_leaf': 2}


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('text',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='',
                                                                                 strategy='constant')),
                                                                  ('combiner',
                                                                   MakeCorpus()),
                                                                  ('tfidf',
                                                                   TfidfVectorizer(max_df=0.9,
                                                                                   max_features=500,
                                                                                   min_df=3,
                                                                                   stop_words='english'))]),
                                                  ['best_tasks_free',
                                                   'subopt_tasks_free',
                                                   'verify_method_free']),
                                                 ('ord',
                                                  Pipeline(steps=[('s...
                                                                                                          'debugging '
                                                                                                          'code',
                                                                                                          'Data '
                                                                                                          'processing '
                                                                                                          'or '
                                                                                                          'analysis',
                                                                                                          'Math '
                                                                                                          'computations'])),
                                                                  ('simpleimputer',
                                                                   SimpleImputer(fill_value=0,
                                                                                 strategy='constant'))]),
                                                  ['best_tasks_select',
                                                   'subopt_tasks_select'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced', max_depth=10,
                                        min_samples_leaf=2, min_samples_split=8,
                                        n_estimators=500, n_jobs=-1,
                                        random_state=22))])

In [136]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Split train into train_small and validation
x_train_small, x_val, y_train_small, y_val = train_test_split(
    x_train, y_train, 
    test_size=0.15,      # 15% validation
    random_state=40,
    stratify=y_train     # important for classification
)

# Fit model normally
best_pipeline.fit(x_train_small, y_train_small)

# Predict validation performance
y_pred_val = best_pipeline.predict(x_val)

val_acc = accuracy_score(y_val, y_pred_val)
print("Validation accuracy:", val_acc)

Validation accuracy: 0.632183908045977


In [10]:
# TF=IDF DEVELOPMENT IN ISOLATION
"""CORPUS:
├── Document 1 (Row 1): col1 + col2 + col3 → Label: ChatGPT
├── Document 2 (Row 2): col1 + col2 + col3 → Label: Claude

Term frequency (TF): the raw count of a term in a document
Inverse document frequency (IDF): how important a word is, idf(t,D)=log(N/df(t))
"""
# GridSearchCV for hyperparam tuning
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

# TODO: TAKE OUT WHEN YOU MOVE TO PIPELINE
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='constant', fill_value='')
df[text_cols] = imputer.fit_transform(df[text_cols])

# make corpus, create new column
df['corpus'] = df[text_cols].agg(' '.join, axis=1)
corpus = df['corpus']
labels = df['target']

# Create pipeline
pipeline = Pipeline([
    # TODO: also try removing stop words and adding max_df
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000))
])

# Define parameter grid
param_grid = {
    'tfidf__max_features': [500, 1000, 2000, 3000, None], # max number of features
    'tfidf__ngram_range': [(1, 1), (1, 2), (1, 3)], # length of context range
    'tfidf__min_df': [1, 2, 3, 4, 5] # minimum number times word has to appear to be included
}

# Search
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy')
grid_search.fit(corpus, labels)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best accuracy: {grid_search.best_score_:.3f}")




Best parameters: {'tfidf__max_features': None, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 3)}
Best accuracy: 0.605


WON'T BE RUNNING THIS BELOW

In [ ]:
# LLM EMBEDDINGS

import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np
import pandas as pd
# use pretrained weights

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

def embed_texts(text_col, batch_size=32):
    """
    Accepts a pandas Series, list, numpy array, or single string and returns
    a numpy array of mean-pooled embeddings (N, hidden_size).
    Replaces missing values with empty strings so the tokenizer always gets str inputs.
    """
    # normalize input to a list of strings
    texts = [ '' if pd.isna(x) else str(x) for x in text_col ]

    if len(texts) == 0:
        return np.zeros((0, model.config.hidden_size))

    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )

        # pass to model with no gradient bc we are not training, just getting embeddings
        with torch.no_grad():
            # done per response in batch  ============================================
            outputs = model(**enc) # outputs.last_hidden_state: (B, seq_len, 768)

            # Mean Pooling: average vectors of each true token vector in the text ====
            mask = enc["attention_mask"].unsqueeze(-1) # (B, seq_len, 1), add 1 dim for multiplication
            masked = outputs.last_hidden_state * mask # ignores padding tokens
            summed = masked.sum(dim=1)
            counts = mask.sum(dim=1).clamp(min=1e-9)
            mean_pooled = summed / counts  # (B, hidden)

            all_embeddings.append(mean_pooled.cpu().numpy())

    return np.vstack(all_embeddings)

# embed_best_tasks_free = embed_texts(df['best_tasks_free'])
# print(embed_best_tasks_free)
# # embed_subopt_tasks_free = embed_texts(df['subopt_tasks_free'])
# embed_verify_method_free = embed_texts(df['verify_method_free'])

def embed_texts(text_cols, df, batch_size=32):
    col_embedding = []
    for col in text_cols:
        embed_texts(df[col], batch_size=batch_size)
        col_embedding.append(col)
    return np.hstack(col_embedding)

print(embed_texts(text_cols, df))


c:\Users\carol\Desktop\Characterize_GenAIModel\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 